# Step 0: Install Dependencies

In [1]:
# Run once — restart kernel after installation if needed
import subprocess, sys
pkgs = [
    "rank_bm25", "faiss-cpu", "sentence-transformers",
    "pandas", "scikit-learn", "pythainlp", "requests", "tqdm"
]
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q"] + pkgs)
print("✅ All packages installed")

✅ All packages installed


# Step 0.1: Imports & Global Config

In [2]:
import os, re, json, math, time, warnings, zipfile, requests
import numpy as np
import pandas as pd
from pathlib import Path
from typing import List, Dict, Tuple, Optional, Any

from sklearn.preprocessing import MinMaxScaler
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer
import faiss

try:
    from pythainlp.tokenize import word_tokenize as thai_tokenize
    THAI_TOKENIZER_AVAILABLE = True
    print("✅ PyThaiNLP loaded — using Thai-aware tokenizer")
except ImportError:
    THAI_TOKENIZER_AVAILABLE = False
    print("⚠️  PyThaiNLP not found — falling back to whitespace tokenizer")

warnings.filterwarnings("ignore")
print("✅ Imports OK")

/opt/anaconda3/envs/nlp-env/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✅ PyThaiNLP loaded — using Thai-aware tokenizer
✅ Imports OK


# Config

In [1]:
# ════════════════════════════════════════════════════════
# ⚙️  GLOBAL CONFIGURATION — edit freely
# ════════════════════════════════════════════════════════
CONFIG = {
    # ── Paths ──────────────────────────────────────────────
    # FIX 1: zip extracts to data/ so KB is data/knowledge_base
    #         and questions file is data/questions.csv  (NOT question.csv)
    "zip_file":           "data.zip",
    "knowledge_base_dir": "data/knowledge_base",
    "questions_file":     "data/questions.csv",      # ← was "data/question.csv"
    "output_file":        "data/submission.csv",

    # ── Chunking ───────────────────────────────────────────
    "chunk_size":    400,
    "chunk_overlap": 80,

    # ── Retrieval ──────────────────────────────────────────
    "vector_top_k": 10,
    "bm25_top_k":   10,
    "rrf_k":        60,
    "final_top_k":   5,

    # ── Embedding ──────────────────────────────────────────
    "embedding_model": "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2",

    # ── Thai LLM ───────────────────────────────────────────
    # Switch: "typhoon" | "openthaigpt" | "pathumma" | "kbtg"
    "llm_model": "typhoon",
    "llm_endpoints": {
        "typhoon": {
            "url":     "https://api.opentyphoon.ai/v1/chat/completions",
            "model":   "typhoon-v2-70b-instruct",
            "api_key": "weWaVg5YOfGjHJpWOxFEdVIWHoOZbyip",   # ← put your key here, or set env TYPHOON_API_KEY
        },
        "openthaigpt": {
            "url":     "https://api.iapp.co.th/openthaigpt/v1/chat/completions",
            "model":   "openthaigpt-1.5-72b-instruct",
            "api_key": "weWaVg5YOfGjHJpWOxFEdVIWHoOZbyip",
        },
        "pathumma": {
            "url":     "https://api.aiforthai.in.th/pathumma/v1/chat/completions",
            "model":   "pathumma-llm-v2",
            "api_key": "weWaVg5YOfGjHJpWOxFEdVIWHoOZbyip",
        },
        "kbtg": {
            "url":     "https://api.kbtg.tech/ai/llm/v1/chat/completions",
            "model":   "kbtg-llm-thai",
            "api_key": "weWaVg5YOfGjHJpWOxFEdVIWHoOZbyip",
        },
    },
    "max_tokens":   512,
    "temperature":  0.1,
}

NOT_FOUND_MSG = "ไม่พบข้อมูล"

# ── Convenience: set API key without touching CONFIG dict ──
# os.environ["TYPHOON_API_KEY"] = "your_key_here"

print("✅ CONFIG loaded")
print(f"   LLM model  : {CONFIG['llm_model']}")
print(f"   KB dir     : {CONFIG['knowledge_base_dir']}")
print(f"   Questions  : {CONFIG['questions_file']}")

✅ CONFIG loaded
   LLM model  : typhoon
   KB dir     : data/knowledge_base
   Questions  : data/questions.csv


# Helper: check_data_integrity()

In [4]:
def check_data_integrity(config: Dict) -> bool:
    """
    Validate that all required paths exist and are non-empty.
    Returns True if everything is OK, False otherwise.
    """
    print("\n🔍 check_data_integrity()")
    ok = True

    # 1. ZIP file
    zip_path = Path(config["zip_file"])
    if zip_path.exists():
        print(f"   ✅ ZIP found : {zip_path}  ({zip_path.stat().st_size:,} bytes)")
    else:
        print(f"   ⚠️  ZIP not found: {zip_path}  (OK if already extracted)")

    # 2. Knowledge-base directory
    kb_path = Path(config["knowledge_base_dir"])
    if not kb_path.exists():
        print(f"   ❌ KB dir missing : {kb_path}")
        ok = False
    else:
        subfolders = [d for d in kb_path.iterdir() if d.is_dir()]
        print(f"   ✅ KB dir found  : {kb_path}")
        print(f"      Subfolders ({len(subfolders)}): {[d.name for d in subfolders]}")
        for sf in subfolders:
            files = list(sf.iterdir())
            print(f"      └─ {sf.name}: {len(files)} file(s)")
            for f in files[:3]:
                print(f"         • {f.name}")
            if len(files) > 3:
                print(f"         ... +{len(files)-3} more")
        total = sum(len(list(sf.iterdir())) for sf in subfolders)
        if total == 0:
            print("   ❌ KB dir is completely empty!")
            ok = False

    # 3. Questions file
    q_path = Path(config["questions_file"])
    if not q_path.exists():
        print(f"   ❌ Questions file missing: {q_path}")
        ok = False
    else:
        try:
            df = pd.read_csv(q_path, nrows=2)
            print(f"   ✅ Questions file: {q_path}  (cols: {df.columns.tolist()})")
        except Exception as e:
            print(f"   ❌ Cannot read questions file: {e}")
            ok = False

    print(f"\n   {'✅ All checks passed' if ok else '❌ Some checks FAILED — fix before proceeding'}")
    return ok

# Step 1: Extract ZIP + Load Documents

In [5]:
# ── FIX 2: Auto-extract the zip if KB dir doesn't exist yet ──────
def extract_zip_if_needed(config: Dict) -> None:
    kb_path  = Path(config["knowledge_base_dir"])
    zip_path = Path(config["zip_file"])

    if kb_path.exists():
        n = sum(1 for _ in kb_path.rglob("*") if Path(_).is_file())
        if n > 0:
            print(f"✅ KB already extracted ({n} files found) — skipping unzip")
            return

    if not zip_path.exists():
        raise FileNotFoundError(
            f"Neither KB dir '{kb_path}' nor zip '{zip_path}' found.\n"
            "Please place data.zip in the same folder as this notebook."
        )

    print(f"📦 Extracting {zip_path} ...")
    with zipfile.ZipFile(zip_path, "r") as z:
        z.extractall(".")
    print(f"✅ Extracted to current directory")
    # Show what was created
    for p in sorted(Path("data").rglob("*"))[:10]:
        print(f"   {p}")


# ── FIX 3: Correct subfolder name is 'products' not 'product' ──
# ── FIX 4: Questions file is 'questions.csv' not 'question.csv' ─
# Both are handled by the corrected CONFIG above.

def load_documents(kb_dir: str) -> List[Dict]:
    """
    Recursively load all .txt / .md files from the knowledge-base.
    Each document → {text, source, filename, subfolder}
    """
    kb_path = Path(kb_dir)

    # ── Debug: show what we find ─────────────────────────────
    print(f"\n📂 [Step 1] Scanning: {kb_path.resolve()}")
    if not kb_path.exists():
        print(f"   ❌ Directory does not exist: {kb_path}")
        print("   → Call extract_zip_if_needed(CONFIG) first!")
        raise FileNotFoundError(f"Knowledge base not found: {kb_dir}")

    subfolders = [d for d in sorted(kb_path.iterdir()) if d.is_dir()]
    print(f"   Detected subfolders: {[s.name for s in subfolders]}")
    for sf in subfolders:
        files = list(sf.iterdir())
        sample = [f.name for f in files[:3]]
        print(f"   └─ {sf.name}: {len(files)} file(s)  sample={sample}")

    # ── FIX 5: Only load .txt and .md — do NOT recurse into
    #    questions.csv / sample_submission.csv as KB documents ──
    SUPPORTED = {".txt", ".md"}
    documents  = []

    for file_path in sorted(kb_path.rglob("*")):
        if not file_path.is_file():
            continue
        if file_path.suffix.lower() not in SUPPORTED:
            print(f"   ⚠️  Skipping unsupported type: {file_path.name}")
            continue
        try:
            text = file_path.read_text(encoding="utf-8").strip()
        except UnicodeDecodeError:
            try:
                text = file_path.read_text(encoding="tis-620").strip()
                print(f"   ℹ️  Read with TIS-620 encoding: {file_path.name}")
            except Exception as e:
                print(f"   ❌ Encoding error, skipping {file_path.name}: {e}")
                continue
        except Exception as e:
            print(f"   ❌ Cannot read {file_path.name}: {e}")
            continue

        if not text:
            print(f"   ⚠️  Empty file, skipping: {file_path.name}")
            continue

        documents.append({
            "text":      text,
            "source":    str(file_path),
            "filename":  file_path.name,
            "subfolder": file_path.parent.name,
        })

    print(f"\n   ✅ Loaded {len(documents)} document(s)")
    if len(documents) == 0:
        print("   ❌ CRITICAL: 0 documents loaded — pipeline cannot continue!")
    else:
        for doc in documents[:5]:
            preview = doc["text"][:80].replace("\n", " ")
            print(f"   [{doc['subfolder']}] {doc['filename']}")
            print(f"     → \"{preview}...\"")
        if len(documents) > 5:
            print(f"   ... and {len(documents)-5} more")
    return documents

# Step 2: Chunking

In [6]:
def chunk_documents(
    documents: List[Dict],
    chunk_size: int    = 400,
    chunk_overlap: int = 80,
) -> List[Dict]:
    """Sliding-window character-level chunking."""
    if not documents:
        print("\n⚠️  [Step 2] No documents to chunk — stopping early.")
        return []

    chunks    = []
    chunk_id  = 0

    for doc in documents:
        text = doc["text"]
        if len(text) <= chunk_size:
            chunks.append({**doc, "chunk_id": chunk_id, "chunk_text": text})
            chunk_id += 1
            continue

        start = 0
        while start < len(text):
            end = min(start + chunk_size, len(text))
            if end < len(text):
                boundary = text.rfind(" ", start, end)
                if boundary > start:
                    end = boundary
            chunk_text = text[start:end].strip()
            if chunk_text:
                chunks.append({**doc, "chunk_id": chunk_id, "chunk_text": chunk_text})
                chunk_id += 1
            start = end - chunk_overlap
            if start >= len(text):
                break

    print(f"\n✂️  [Step 2] Created {len(chunks)} chunk(s)"
          f"  (chunk_size={chunk_size}, overlap={chunk_overlap})")

    if len(chunks) == 0:
        print("   ❌ CRITICAL: 0 chunks created — check document content!")
        return []

    for c in chunks[:3]:
        preview = c["chunk_text"][:70].replace("\n", " ")
        print(f"   chunk_{c['chunk_id']} [{c['subfolder']}]: \"{preview}...\"")
    print(f"   ... total {len(chunks)} chunks across {len(documents)} docs")
    return chunks

# Step 3: Build Vector Index (FAISS)

In [7]:
def build_vector_index(
    chunks: List[Dict],
    model_name: str,
) -> Tuple[Optional[faiss.Index], Optional[SentenceTransformer], Optional[np.ndarray]]:
    """Encode chunks → FAISS IndexFlatIP (cosine via unit-norm)."""

    # ── FIX 6: Guard against empty chunk list ────────────────
    if not chunks:
        print("\n❌ [Step 3] Cannot build index — chunks list is EMPTY.")
        print("   Fix Steps 1 & 2 first, then re-run.")
        return None, None, None

    print(f"\n🔢 [Step 3] Building vector index")
    print(f"   Model  : {model_name}")
    print(f"   Chunks : {len(chunks)}")

    try:
        embed_model = SentenceTransformer(model_name)
    except Exception as e:
        print(f"   ❌ Failed to load embedding model: {e}")
        print("   Check your internet connection or model name.")
        return None, None, None

    texts = [c["chunk_text"] for c in chunks]

    try:
        embeddings = embed_model.encode(
            texts,
            batch_size=32,
            show_progress_bar=True,
            normalize_embeddings=True,
        )
        embeddings = np.array(embeddings, dtype=np.float32)
    except Exception as e:
        print(f"   ❌ Embedding failed: {e}")
        return None, None, None

    # ── FIX 7: Validate shape before FAISS add ──────────────
    if embeddings.ndim != 2 or embeddings.shape[0] == 0:
        print(f"   ❌ Bad embedding shape: {embeddings.shape}")
        return None, None, None

    dim   = embeddings.shape[1]
    index = faiss.IndexFlatIP(dim)
    index.add(embeddings)

    print(f"   ✅ Index built: {index.ntotal} vectors, dim={dim}")
    return index, embed_model, embeddings

# Step 4: Vector Search

In [8]:
def vector_search(
    query: str,
    faiss_index: faiss.Index,
    embed_model: SentenceTransformer,
    chunks: List[Dict],
    top_k: int = 10,
) -> List[Dict]:
    if not chunks or faiss_index is None or faiss_index.ntotal == 0:
        return []

    # FIX 8: cap top_k to avoid FAISS padding with -1
    safe_k = min(top_k, faiss_index.ntotal)

    q_emb = embed_model.encode([query], normalize_embeddings=True).astype(np.float32)
    if q_emb.ndim == 1:
        q_emb = q_emb.reshape(1, -1)

    scores, indices = faiss_index.search(q_emb, safe_k)

    # FIX 9: unpack row 0 safely regardless of array dimensionality
    idx_row   = indices[0] if indices.ndim == 2 else indices
    score_row = scores[0]  if scores.ndim  == 2 else scores

    results = []
    for rank, (idx, score) in enumerate(zip(idx_row, score_row)):
        idx = int(idx)
        if idx < 0 or idx >= len(chunks):
            continue
        results.append({"chunk": chunks[idx], "score": float(score), "rank": rank})
    return results

# Step 5: BM25 Search

In [9]:
def _tokenize(text: str) -> List[str]:
    if THAI_TOKENIZER_AVAILABLE:
        return thai_tokenize(text, engine="newmm", keep_whitespace=False)
    tokens = []
    for word in text.split():
        if re.search(r'[\u0E00-\u0E7F]', word):
            tokens.extend(list(word))
        else:
            tokens.append(word.lower())
    return [t for t in tokens if t.strip()]


def build_bm25_index(chunks: List[Dict]) -> Optional[BM25Okapi]:
    if not chunks:
        print("⚠️  build_bm25_index: empty chunk list")
        return None
    corpus = [_tokenize(c["chunk_text"]) for c in chunks]
    return BM25Okapi(corpus)


def bm25_search(
    query: str,
    bm25_index: BM25Okapi,
    chunks: List[Dict],
    top_k: int = 10,
) -> List[Dict]:
    if not chunks or bm25_index is None:
        return []

    q_tokens = _tokenize(query)
    if not q_tokens:
        return []

    scores  = bm25_index.get_scores(q_tokens)
    safe_k  = min(top_k, len(scores))
    top_idx = np.argsort(scores)[::-1][:safe_k]

    results = []
    for rank, idx in enumerate(top_idx):
        idx = int(idx)
        if idx >= len(chunks) or scores[idx] <= 0:
            continue
        results.append({"chunk": chunks[idx], "score": float(scores[idx]), "rank": rank})
    return results

# Step 6: Hybrid Fusion (RRF)

In [10]:
def hybrid_fusion(
    vector_results: List[Dict],
    bm25_results:   List[Dict],
    rrf_k: int = 60,
) -> List[Dict]:
    rrf_scores: Dict[int, Dict] = {}

    def _add(results, source):
        for item in results:
            cid = item["chunk"]["chunk_id"]
            if cid not in rrf_scores:
                rrf_scores[cid] = {
                    "chunk": item["chunk"], "rrf_score": 0.0,
                    "vector_score": None, "bm25_score": None,
                    "vector_rank":  None, "bm25_rank":  None,
                }
            rrf_scores[cid]["rrf_score"] += 1.0 / (rrf_k + item["rank"] + 1)
            rrf_scores[cid][f"{source}_score"] = item["score"]
            rrf_scores[cid][f"{source}_rank"]  = item["rank"]

    _add(vector_results, "vector")
    _add(bm25_results,   "bm25")

    return sorted(rrf_scores.values(), key=lambda x: x["rrf_score"], reverse=True)

# Step 7: Reranking

In [11]:
def rerank(query: str, fused: List[Dict], embed_model) -> List[Dict]:
    if not fused or embed_model is None:
        return fused

    chunk_texts = [item["chunk"]["chunk_text"] for item in fused]
    try:
        q_emb  = embed_model.encode([query], normalize_embeddings=True)
        c_embs = embed_model.encode(chunk_texts, normalize_embeddings=True)
        if q_emb.ndim == 1:
            q_emb = q_emb.reshape(1, -1)
        if c_embs.ndim == 1:
            c_embs = c_embs.reshape(1, -1)
        cos_scores = np.dot(c_embs, q_emb[0])
    except Exception as e:
        print(f"   ⚠️  rerank failed: {e} — using RRF score only")
        for item in fused:
            item["rerank_score"] = item["rrf_score"]
            item["cosine_score"] = 0.0
        return fused

    mn, mx     = cos_scores.min(), cos_scores.max()
    norm_cos   = (cos_scores - mn) / (mx - mn + 1e-9)
    rrf_arr    = np.array([x["rrf_score"] for x in fused])
    mn_r, mx_r = rrf_arr.min(), rrf_arr.max()
    norm_rrf   = (rrf_arr - mn_r) / (mx_r - mn_r + 1e-9)
    combined   = 0.5 * norm_cos + 0.5 * norm_rrf

    for i, item in enumerate(fused):
        item["rerank_score"] = float(combined[i])
        item["cosine_score"] = float(cos_scores[i])

    return sorted(fused, key=lambda x: x["rerank_score"], reverse=True)

# Step 8: Top-K Selection

In [12]:
def select_top_k(reranked: List[Dict], k: int = 5) -> List[Dict]:
    return reranked[:k]

# Step 9: Prompt Template

In [13]:
def build_prompt(query: str, top_chunks: List[Dict], choices: Optional[Dict] = None) -> str:
    """
    Build a Thai RAG prompt.
    If multiple-choice options are provided, instruct the LLM to return
    only the choice NUMBER (1-10).
    """
    if not top_chunks:
        context_text = "ไม่มีข้อมูลที่เกี่ยวข้อง"
    else:
        parts = []
        for i, item in enumerate(top_chunks, 1):
            src = item["chunk"].get("subfolder", "unknown")
            fn  = item["chunk"].get("filename",  "unknown")
            txt = item["chunk"]["chunk_text"]
            parts.append(f"[{i}] (แหล่งที่มา: {src}/{fn})\n{txt}")
        context_text = "\n\n".join(parts)

    if choices:
        choice_lines = "\n".join(
            f"  ตัวเลือกที่ {k}: {v}" for k, v in choices.items()
        )
        task = (
            f"\nตัวเลือกคำตอบ:\n{choice_lines}\n\n"
            "ให้ตอบเฉพาะ ตัวเลข ของตัวเลือกที่ถูกต้อง (เช่น 1, 2, 3, ...) เท่านั้น "
            "ห้ามตอบข้อความอื่น"
        )
    else:
        task = "\nคำตอบ (ตอบเป็นภาษาไทย อ้างอิงจาก Context เท่านั้น):"

    prompt = f"""คุณเป็นผู้ช่วยอัจฉริยะที่ตอบคำถามโดยอ้างอิงจากข้อมูลที่ให้มาเท่านั้น
ห้ามใช้ความรู้ภายนอก หากไม่พบคำตอบใน Context ให้ตอบว่า "ไม่พบข้อมูล"

===== Context =====
{context_text}
===================

คำถาม: {query}{task}"""
    return prompt

# Step 10: Generate Answer (Thai LLM)

In [14]:
def generate_answer(prompt: str, config: Dict) -> str:
    """Call the configured Thai LLM. Returns answer string or NOT_FOUND_MSG."""
    model_key = config["llm_model"]
    ep        = config["llm_endpoints"].get(model_key)
    if ep is None:
        print(f"   ❌ Unknown model: {model_key}")
        return NOT_FOUND_MSG

    # FIX 10: api_key is stored directly in ep dict (not via env var name)
    api_key = ep.get("api_key", "").strip()
    # Also try environment variable as fallback
    if not api_key:
        env_name = f"{model_key.upper()}_API_KEY"
        api_key  = os.environ.get(env_name, "").strip()
    if not api_key:
        print(f"   ❌ No API key for '{model_key}'. "
              f"Set CONFIG['llm_endpoints']['{model_key}']['api_key'] = 'YOUR_KEY'")
        return NOT_FOUND_MSG

    headers = {"Authorization": f"Bearer {api_key}", "Content-Type": "application/json"}
    payload = {
        "model":       ep["model"],
        "messages":    [{"role": "user", "content": prompt}],
        "max_tokens":  config["max_tokens"],
        "temperature": config["temperature"],
    }
    try:
        resp = requests.post(ep["url"], headers=headers, json=payload, timeout=60)
        resp.raise_for_status()
        answer = resp.json()["choices"][0]["message"]["content"].strip()
        return answer if answer else NOT_FOUND_MSG
    except requests.exceptions.HTTPError as e:
        print(f"   ❌ HTTP {resp.status_code}: {e}")
    except Exception as e:
        print(f"   ❌ LLM error: {e}")
    return NOT_FOUND_MSG

# Full Pipeline — run_pipeline()

In [15]:
def run_pipeline(
    query:       str,
    chunks:      List[Dict],
    faiss_index,
    embed_model,
    bm25_index,
    config:      Dict,
    choices:     Optional[Dict] = None,
    debug:       bool = True,
) -> str:
    sep = "─" * 60
    if debug:
        print(f"\n{'='*60}")
        print(f"  QUESTION: {query}")
        if choices:
            for k, v in choices.items():
                print(f"    [{k}] {v}")
        print(f"{'='*60}")

    # Step 4
    vec_results = vector_search(query, faiss_index, embed_model, chunks, config["vector_top_k"])
    if debug:
        print(f"\n[Step 4] Vector Search — {len(vec_results)} results")
        for r in vec_results[:3]:
            print(f"   rank={r['rank']+1}  score={r['score']:.4f}  "
                  f"\"{r['chunk']['chunk_text'][:60].replace(chr(10),' ')}...\"")

    # Step 5
    bm25_results = bm25_search(query, bm25_index, chunks, config["bm25_top_k"])
    if debug:
        print(f"\n[Step 5] BM25 Search — {len(bm25_results)} results")
        for r in bm25_results[:3]:
            print(f"   rank={r['rank']+1}  score={r['score']:.4f}  "
                  f"\"{r['chunk']['chunk_text'][:60].replace(chr(10),' ')}...\"")

    # Step 6
    fused = hybrid_fusion(vec_results, bm25_results, config["rrf_k"])
    if debug:
        print(f"\n[Step 6] Hybrid Fusion — {len(fused)} unique candidates")

    # Step 7
    reranked = rerank(query, fused, embed_model)
    if debug:
        print(f"\n[Step 7] Reranked — top 3:")
        for item in reranked[:3]:
            print(f"   rerank={item['rerank_score']:.4f}  "
                  f"\"{item['chunk']['chunk_text'][:60].replace(chr(10),' ')}...\"")

    # Step 8
    top_chunks = select_top_k(reranked, config["final_top_k"])
    if debug:
        print(f"\n[Step 8] Top-{config['final_top_k']} selected:")
        for i, item in enumerate(top_chunks, 1):
            src = item['chunk'].get('subfolder','?')
            fn  = item['chunk'].get('filename','?')
            print(f"   [{i}] {src}/{fn}  rerank={item['rerank_score']:.4f}")
            print(f"        \"{item['chunk']['chunk_text'][:80].replace(chr(10),' ')}...\"")

    # Step 9
    prompt = build_prompt(query, top_chunks, choices=choices)
    if debug:
        print(f"\n[Step 9] Prompt preview (400 chars):")
        print(f"   {prompt[:400]}...")

    # Step 10
    answer = generate_answer(prompt, config)
    if debug:
        print(f"\n[Step 10] Answer: {answer}")
        print(sep)

    return answer

# Main Execution

In [ ]:
def main():
    print("\n" + "="*60)
    print("  RAG PIPELINE — THAI LLM")
    print("="*60)

    # ── Extract zip if needed ──────────────────────────────────
    extract_zip_if_needed(CONFIG)

    # ── Integrity check ────────────────────────────────────────
    if not check_data_integrity(CONFIG):
        print("\n❌ Integrity check failed — aborting.")
        return None, None

    # ── Step 1 ─────────────────────────────────────────────────
    documents = load_documents(CONFIG["knowledge_base_dir"])
    if not documents:
        print("\n❌ 0 documents loaded — aborting.")
        return None, None

    # ── Step 2 ─────────────────────────────────────────────────
    chunks = chunk_documents(documents, CONFIG["chunk_size"], CONFIG["chunk_overlap"])
    if not chunks:
        print("\n❌ 0 chunks created — aborting.")
        return None, None

    # ── Step 3 ─────────────────────────────────────────────────
    faiss_index, embed_model, _ = build_vector_index(chunks, CONFIG["embedding_model"])
    if faiss_index is None:
        print("\n❌ Vector index build failed — aborting.")
        return None, None

    bm25_index = build_bm25_index(chunks)
    print(f"\n📊 [Step 3b] BM25 index built over {len(chunks)} chunks")

    # ── Load questions ──────────────────────────────────────────
    q_path = Path(CONFIG["questions_file"])
    if not q_path.exists():
        print(f"\n❌ Questions file not found: {q_path}")
        return None, None

    df_q = pd.read_csv(q_path)
    print(f"\n📋 Loaded {len(df_q)} question(s) from '{q_path}'")
    print(f"   Columns: {df_q.columns.tolist()}")
    print(df_q.head(3).to_string())

    # ── FIX 11: Detect question + id + choice columns ──────────
    q_col     = next((c for c in df_q.columns if "question" in c.lower()), None)
    id_col    = next((c for c in df_q.columns if c.lower() == "id"), None)
    # Choice columns: choice_1 ... choice_10
    choice_cols = [c for c in df_q.columns if re.match(r"choice_\d+", c)]
    print(f"\n   question col : {q_col}")
    print(f"   id col       : {id_col}")
    print(f"   choice cols  : {choice_cols}")

    if q_col is None:
        print("\n❌ Cannot find question column — aborting.")
        return None, None

    # ── Loop ───────────────────────────────────────────────────
    results        = []
    answer_fah_mai = None

    for idx, row in df_q.iterrows():
        question = str(row[q_col]).strip()
        q_id     = str(row[id_col]) if id_col else str(idx)

        # Build choices dict {1: text, 2: text, ...}
        choices = {}
        for cc in choice_cols:
            num = cc.split("_")[1]
            val = row[cc]
            if pd.notna(val):
                choices[int(num)] = str(val)

        print(f"\n{'#'*60}")
        print(f"  Q{idx+1}/{len(df_q)}  id={q_id}: {question}")
        print(f"{'#'*60}")

        answer = run_pipeline(
            query       = question,
            chunks      = chunks,
            faiss_index = faiss_index,
            embed_model = embed_model,
            bm25_index  = bm25_index,
            config      = CONFIG,
            choices     = choices if choices else None,
            debug       = True,
        )

        answer_fah_mai = answer
        results.append({"id": q_id, "answer": answer})

    # ── Output ─────────────────────────────────────────────────
    df_results = pd.DataFrame(results)
    print("\n\n" + "="*60)
    print("  FINAL RESULTS")
    print("="*60)
    print(df_results.to_string(index=False))

    out_path = CONFIG["output_file"]
    df_results.to_csv(out_path, index=False, encoding="utf-8-sig")
    print(f"\n✅ Saved → {out_path}")
    print(f"   answer_fah_mai = \"{answer_fah_mai}\"")
    return df_results, answer_fah_mai

: 

# ▶ Set API Key & Run

In [ ]:
# ── Set your Typhoon API key here ─────────────────────────────
CONFIG["llm_endpoints"]["typhoon"]["api_key"] = "weWaVg5YOfGjHJpWOxFEdVIWHoOZbyip"

# ── Run the full pipeline ──────────────────────────────────────
df_results, answer_fah_mai = main()


  RAG PIPELINE — THAI LLM
✅ KB already extracted (119 files found) — skipping unzip

🔍 check_data_integrity()
   ⚠️  ZIP not found: data.zip  (OK if already extracted)
   ✅ KB dir found  : data/knowledge_base
      Subfolders (3): ['products', 'store_info', 'policies']
      └─ products: 110 file(s)
         • JC-CH-006_judchuam_cable_usbc_to_usbc_1m.md
         • SF-TB-005_saifah_tab_mini_7.md
         • SF-SP-005_saifah_phone_v7_lite.md
         ... +107 more
      └─ store_info: 3 file(s)
         • about_fahmai.md
         • buying_guide_smartphones_2569.md
         • general_faq.md
      └─ policies: 5 file(s)
         • shipping_policy.md
         • membership_points_policy.md
         • warranty_policy.md
         ... +2 more
   ✅ Questions file: data/questions.csv  (cols: ['id', 'question', 'choice_1', 'choice_2', 'choice_3', 'choice_4', 'choice_5', 'choice_6', 'choice_7', 'choice_8', 'choice_9', 'choice_10'])

   ✅ All checks passed

📂 [Step 1] Scanning: /Users/moi/Desktop/CO